# Step 0b — Test Your Setup & Meet Your First Agent

🇬🇧 **English** (this notebook)

A second, hands-on pass at [Step 0](step_00_setup_and_python_basics.ipynb): instead of reading about the tools, this notebook actually **checks** that your environment works, tours the project structure, and has you create and talk to your very first CrewAI agent — before Step 3 introduces the full YAML-config version of the same idea.

**By the end of this notebook, you will:**

- Have confirmed your Python environment, packages, and API keys are all correctly set up
- Know your way around this repo's actual folder structure
- Have created a standalone CrewAI `Agent` and talked to it directly — no YAML, no Task, no Crew required
- Have seen, first-hand, what an agent can't do without help (setting up why Step 5 adds tools and RAG)

## Part 1 — Environment Setup Verification

### 1.1 Python environment

In [ ]:
import os
import sys

print("🐍 Python Environment Check")
print("=" * 40)
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")
print(f"Current working directory: {os.getcwd()}")
print(f"Virtual environment: {os.getenv('VIRTUAL_ENV', 'Not detected')}")

if ".venv" not in sys.executable:
    print("\n⚠️  This doesn't look like this project's .venv — check the kernel picker (top-right) and select \"research_crew\".")
else:
    print("\n✅ Running inside this project's .venv")

### 1.2 Package imports

This project uses `crewai` directly — no LangChain, no separate custom package to install.

In [ ]:
print("📦 Package Import Check")
print("=" * 40)

try:
    from crewai import LLM, Agent, Task, Crew
    print("✅ crewai imported successfully (LLM, Agent, Task, Crew)")

    from dotenv import load_dotenv
    print("✅ python-dotenv imported successfully")

    print("\n🎉 All required packages are available!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("💡 Wrong kernel selected? Pick \"research_crew\" from the kernel picker (top-right).")

### 1.3 Load `.env`

In [ ]:
from dotenv import load_dotenv

load_dotenv()

### 1.4 Environment variables

In [ ]:
print("🔑 Environment Variables Check")
print("=" * 40)

# At least one chat-model key is required; the others are only needed for later steps
# (SERPER_API_KEY for step 5's web search tool, MODEL to override the default model).
required_keys = ["GEMINI_API_KEY"]
optional_keys = ["OPENAI_API_KEY", "SERPER_API_KEY", "MODEL"]

print("Required:")
for key in required_keys:
    value = os.getenv(key)
    if value:
        print(f"✅ {key}: {'*' * 8}...{value[-2:] if len(value) > 4 else '****'}")
    else:
        print(f"❌ {key}: not set — copy .env.example to .env and fill it in")

print("\nOptional (needed for later steps, not this one):")
for key in optional_keys:
    value = os.getenv(key)
    if value:
        print(f"✅ {key}: set")
    else:
        print(f"⚠️  {key}: not set (optional for now)")

## Part 2 — Project Structure Exploration

### 2.1 Directory tree

In [ ]:
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing pyproject.toml is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


def show_directory_tree(path, prefix="", max_depth=2, current_depth=0):
    """Display directory structure as a tree, skipping dotfiles/__pycache__/output/.venv."""
    if current_depth > max_depth:
        return
    path = Path(path)
    skip = {"__pycache__", ".venv", "output", ".git"}
    items = sorted(
        item for item in path.iterdir()
        if not item.name.startswith(".") and item.name not in skip
    )
    for i, item in enumerate(items):
        is_last = i == len(items) - 1
        current_prefix = "└── " if is_last else "├── "
        print(f"{prefix}{current_prefix}{item.name}")
        if item.is_dir() and current_depth < max_depth:
            next_prefix = prefix + ("    " if is_last else "│   ")
            show_directory_tree(item, next_prefix, max_depth, current_depth + 1)


ROOT = find_repo_root(Path.cwd())

print("🏗️ Project Structure")
print("=" * 40)
print(f"{ROOT.name}/")
show_directory_tree(ROOT, max_depth=2)

### 2.2 Key components

Unlike the reference course's custom `agentic_ai` package, this project's "agent code" is CrewAI itself, configured through YAML — there's no framework code of ours to browse, just configuration and a thin `crew.py`.

In [ ]:
print("🔍 Key Components Overview")
print("=" * 40)

components = {
    "src/research_crew/crew.py": "Defines the agents, tasks, and the Crew itself - short on purpose",
    "src/research_crew/config/agents.yaml": "Each agent's role/goal/backstory",
    "src/research_crew/config/tasks.yaml": "Each task's description/expected_output/agent assignment",
    "exercises/en/prompting/": "Steps 1-2 - plain crewai.LLM calls, no agents yet",
    "exercises/en/agents/": "Steps 3-5 - the CrewAI agent/task/crew exercises",
}

for path, description in components.items():
    print(f"📁 {path}")
    print(f"   {description}\n")

## Part 3 — Basic LLM Call

The same `crewai.LLM` class from steps 1-2 — a quick sanity check that your API key actually works before moving on to an agent.

In [ ]:
from crewai import LLM

print("🤖 Setting up the LLM")
print("=" * 40)

if not os.getenv("GEMINI_API_KEY") and not os.getenv("OPENAI_API_KEY"):
    print("❌ Neither GEMINI_API_KEY nor OPENAI_API_KEY is set. Fill in .env first.")
else:
    llm = LLM(model=os.getenv("MODEL", "gemini/gemini-2.5-flash"), temperature=0.7)
    print(f"✅ LLM initialized: {llm.model}")

    response = llm.call(
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Keep responses to one sentence."},
            {"role": "user", "content": "Hello! Explain what you are in one sentence."},
        ]
    )
    print(f"\n💬 Test response: {response}")

## Part 4 — Your First Agent

### 4.1 A standalone `Agent` — no Task, no Crew, no YAML

CrewAI's `Agent` class can run entirely on its own via `.kickoff()` — this is the same class Step 3's `agents.yaml`-configured agents use internally, just called directly instead of through a `Crew`. `role`/`goal`/`backstory` are exactly the components you already used as `persona`/`instruction`/`context` in step 2b.

**Note:** in a notebook, `agent.kickoff(...)` must be awaited — Jupyter's kernel runs its own asyncio event loop, and `kickoff()` detects that and returns a coroutine instead of executing directly (this is meant for use inside CrewAI Flows, but a notebook kernel triggers the same detection). A plain `.py` script run via `uv run python script.py` would not need `await` here — this is purely a notebook-vs-script difference.

In [ ]:
from crewai import Agent

print("🤖 Creating your first Agent")
print("=" * 40)

agent = Agent(
    role="Learning Companion",
    goal="Help the user understand what AI agents are and how they work",
    backstory="You are a friendly assistant, new here to help students learn.",
    llm=llm,
)
print(f"✅ Agent created — role: {agent.role}")

# Jupyter runs its own event loop, and kickoff() detects that automatically and
# returns a coroutine instead of running synchronously - awaiting it is required here
# (this quirk only shows up in a notebook; a plain .py script would not need it).
output = await agent.kickoff("Hello! What's your role and what can you do?")
print(f"\n💬 Agent: {output.raw}")

### 4.2 Customizing personality via `backstory`/`goal`

Same effect you saw in step 2b's persona component, now on a real `Agent` instead of a raw system message string.

In [ ]:
study_buddy = Agent(
    role="StudyBuddy",
    goal="Help students learn about AI and programming",
    backstory=(
        "You are StudyBuddy, an enthusiastic AI learning companion. "
        "Always be encouraging, explain concepts with examples, and ask a "
        "follow-up question to check understanding. Keep responses concise."
    ),
    llm=llm,
)

question = "I'm new to AI agents. Can you explain what you are and how you work?"
print(f"💬 {question}")
output = await study_buddy.kickoff(question)
print(f"\nStudyBuddy: {output.raw}")

### 4.3 Follow-up questions — conversation state isn't automatic

Each `.kickoff()` call starts fresh — it doesn't remember earlier turns by itself. To ask a follow-up that has context, build the messages list yourself (`output.messages` has exactly what the agent saw and said last time), the same pattern you used for the `messages` list in step 2.

In [ ]:
follow_up = "What's the difference between you and a regular chatbot?"

conversation = output.messages + [{"role": "user", "content": follow_up}]
print(f"💬 {follow_up}")

output2 = await study_buddy.kickoff(conversation)
print(f"\nStudyBuddy: {output2.raw}")

### 4.4 What the agent can't do (yet)

No tools are attached to this agent, so anything requiring live data is out of reach — this is exactly the gap Step 5's `SerperDevTool` and RAG knowledge sources close.

In [ ]:
question = (
    "What is 15 * 24? Also, what's the current weather in Cologne? "
    "Explain how you got both answers."
)
print(f"💬 {question}")

output3 = await study_buddy.kickoff(question)
print(f"\nStudyBuddy: {output3.raw}")

**Observations to look for:**

- The arithmetic (15 × 24 = 360) should be correct — the model can do this from reasoning alone, no tool needed.
- The weather question should get an honest "I don't have real-time access" answer, not a made-up temperature.
- This is exactly why step 5 adds tools (live web search) and RAG (retrieval from a document you provide) — both address the same root limitation: a model's knowledge is frozen at training time.

That's it for Step 0 — on to [Step 1 — Zero-Shot Prompting](prompting/step_01_zero_shot_prompting.ipynb).